# Person tracking on `dance.mp4` — reproduction of the CV pipeline stall

**Written for: the NVIDIA engineers reviewing this box.**

## What this notebook does
It walks the VSS **alerts** profile in **`2d_cv`** mode step by step and proves that every stage works **up to the perception engine** — and that the perception engine (`vss-rtvi-cv`, DeepStream) accepts the stream but **never produces a single detection frame**, so nothing lands in Kafka `mdx-raw` or Elasticsearch `mdx-raw-*`.

This is the root cause of our original tracking notebook hanging forever at step 7: it polls ES `mdx-raw-*`, which is never created because no CV output is ever emitted.

## Expected pipeline (2d_cv)
```
upload dance.mp4                                              [works]
  -> vss-agent  POST /api/v1/videos + chunk upload + /complete  [works, HTTP 200]
  -> VST/VIOS stores file, emits camera_streaming webhook       [works, HTTP 200]
  -> vss-rtvi-cv /api/v1/stream/add, downloads file, adds source [works]
  -> DeepStream decode -> nvstreammux -> Grounding DINO -> tracker
  -> Kafka mdx-raw (nv.Frame: objects[].id + bbox)             *** NEVER HAPPENS ***
  -> Logstash -> Elasticsearch mdx-raw-*                        *** NEVER CREATED ***
```

## The symptom (isolated in Step 6)
`vss-rtvi-cv` logs, forever, for the added source:
```
**PERF:  0.00000 (0.00000)  source_id : 0  stream_name dance...  sensor_id ...
Active sources : 0
```
with repeated `nvv4l2decoder0:sink: Got data flow before segment event`. Frames reach the decoder but never propagate downstream, so `Active sources` stays `0` and PERF stays `0 FPS`.

> Run the cells top to bottom. Each step prints PASS/FAIL evidence. Steps 1-5 should PASS; Step 6 is where it FAILS.

## 0. Config & helpers
Endpoints of the running alerts/2d_cv stack, plus small helpers to run `docker`/`curl` and read Kafka offsets. Nothing here mutates the system.

In [1]:
import json, subprocess, time, uuid, mimetypes
from pathlib import Path
import requests

AGENT_URL   = "http://127.0.0.1:8000"       # vss-agent
ES_URL      = "http://127.0.0.1:9200"       # elasticsearch
RTVI_CV_URL = "http://127.0.0.1:9010"       # vss-rtvi-cv (perception)
KAFKA_CTR   = "kafka"                        # kafka container name
RTVI_CV_CTR = "vss-rtvi-cv"                  # perception container name
VIDEO       = Path.home() / "Documents/video/dance.mp4"
UPLOAD_TS   = "2025-01-01T00:00:00"

def sh(cmd, timeout=60):
    """Run a shell command, return (rc, stdout+stderr)."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, (p.stdout + p.stderr)

def kafka_offset(topic):
    """Total messages across all partitions of a Kafka topic (0 if empty/missing)."""
    rc, out = sh(f"docker exec {KAFKA_CTR} kafka-get-offsets --bootstrap-server localhost:9092 --topic {topic}")
    total = 0
    for line in out.splitlines():
        parts = line.strip().split(":")
        if len(parts) == 3 and parts[2].isdigit():
            total += int(parts[2])
    return total

def rtvi_logs(since=None):
    """rtvi-cv container logs; since=None returns the FULL log (needed for one-time startup lines)."""
    flag = f"--since {since}" if since else ""
    rc, out = sh(f"docker logs {RTVI_CV_CTR} {flag} 2>&1")
    return out

print("config loaded; video exists:", VIDEO.exists(), VIDEO)

config loaded; video exists: True /home/nvidia/Documents/video/dance.mp4


## 1. Health check — the alerts/2d_cv stack is up
Confirms the containers that matter for CV tracking are running, and that the agent + Elasticsearch answer.

In [2]:
need = ["vss-agent", "vss-rtvi-cv", "vss-vios-ingress", "vss-vios-nvstreamer",
        "vss-vios-streamprocessing", "kafka", "elasticsearch", "logstash", "vss-behavior-analytics"]
rc, out = sh("docker ps --format '{{.Names}}\t{{.Status}}'")
running = {l.split(chr(9))[0]: l.split(chr(9))[1] for l in out.splitlines() if chr(9) in l}
for n in need:
    print(f"{'PASS' if n in running else 'MISSING':8} {n:32} {running.get(n,'')}")

print("\nagent /health:", end=" ")
try:
    print(requests.get(f"{AGENT_URL}/health", timeout=10).status_code)
except Exception as e:
    try: print("openapi", requests.get(f"{AGENT_URL}/openapi.json", timeout=10).status_code)
    except Exception as e2: print("unreachable", e2)
print("elasticsearch:", requests.get(f"{ES_URL}", timeout=10).status_code)

PASS     vss-agent                        Up 11 hours (healthy)
PASS     vss-rtvi-cv                      Up 11 hours
PASS     vss-vios-ingress                 Up 11 hours (healthy)
PASS     vss-vios-nvstreamer              Up 11 hours
PASS     vss-vios-streamprocessing        Up 11 hours (healthy)
PASS     kafka                            Up 11 hours (healthy)
PASS     elasticsearch                    Up 11 hours (healthy)
PASS     logstash                         Up 11 hours
PASS     vss-behavior-analytics           Up 11 hours

agent /health: 200
elasticsearch: 200


## 2. The perception engine has its models loaded
`vss-rtvi-cv` (DeepStream) must have the Grounding DINO detector (Triton `ensemble_python_gdino`) and the ReID tracker (`resnet50_market1501`) initialized.

> These lines are printed **once, at container startup**, so we search the **full** container log (not a recent time window). If the container has been up for hours, a `--since 10m` search would wrongly report NOT FOUND. Presence here means a later stall is **not** a model-loading problem.

In [3]:
logs = rtvi_logs()  # full log — model-load lines are one-time startup events
checks = {
    "Grounding DINO (Triton ensemble)": "ensemble_python_gdino",
    "Tracker engine (ReID resnet50_market1501)": "resnet50_market1501",
    "Tracker initialized": "[NvMultiObjectTracker] Initialized",
}
for label, needle in checks.items():
    print(f"{'PASS' if needle in logs else 'NOT FOUND':10} {label}")

PASS       Grounding DINO (Triton ensemble)
PASS       Tracker engine (ReID resnet50_market1501)
PASS       Tracker initialized


## 3. Baseline — nothing has been produced yet
Before uploading, record that Kafka `mdx-raw` is empty and Elasticsearch has no `mdx-raw-*` index. This is the "before" side of the reproduction.

In [4]:
print("Kafka mdx-raw messages :", kafka_offset("mdx-raw"))
print("Kafka mdx-frames       :", kafka_offset("mdx-frames"))
rc, out = sh(f"curl -s '{ES_URL}/_cat/indices?h=index,docs.count'")
mdx_idx = [l for l in out.splitlines() if 'mdx-raw' in l]
print("ES mdx-raw-* indices   :", mdx_idx or "(none)")

Kafka mdx-raw messages : 0
Kafka mdx-frames       : 0
ES mdx-raw-* indices   : (none)


## 4. Ingest `dance.mp4` (the proven upload recipe)
Agent-mediated 3-step upload, exactly as our working client does it:
1. `POST {agent}/api/v1/videos` `{filename}` -> returns a VST upload URL.
2. Single-chunk `POST` to that URL with the `nvstreamer-*` headers (the whole file as one chunk).
3. `POST {agent}/api/v1/videos/{sensorId}/complete` -> this is what triggers VIOS to notify `vss-rtvi-cv`.

> Note: `dance.mp4` carries an AAC audio track. `vss-rtvi-cv` logs `No decoder available for audio/mpeg (AAC)`. We upload the file **as-is** here to reproduce the real path; whether audio is the trigger is an open question for Step 7.

In [5]:
def upload(src: Path):
    fn = f"{src.stem}_{uuid.uuid4().hex[:8]}.mp4"
    mime = mimetypes.guess_type(fn)[0] or "video/mp4"
    # 1. init
    r = requests.post(f"{AGENT_URL}/api/v1/videos", json={"filename": fn}, timeout=(15, 60))
    r.raise_for_status(); url = r.json()["url"]
    # 2. single-chunk upload
    with src.open("rb") as h:
        s = requests.post(url, headers={
                "nvstreamer-chunk-number": "1", "nvstreamer-total-chunks": "1",
                "nvstreamer-is-last-chunk": "true", "nvstreamer-identifier": uuid.uuid4().hex,
                "nvstreamer-file-name": fn},
            files={"mediaFile": (fn, h, mime)},
            data={"filename": fn, "metadata": json.dumps({"timestamp": UPLOAD_TS})},
            timeout=(15, 900))
    s.raise_for_status(); sid = s.json()["sensorId"]
    # 3. complete -> fans out to rtvi-cv
    p = s.json(); p["filename"] = fn
    c = requests.post(f"{AGENT_URL}/api/v1/videos/{sid}/complete", json=p, timeout=(15, 120))
    return {"sensorId": sid, "name": Path(fn).stem,
            "upload_status": s.status_code, "complete_status": c.status_code,
            "complete_body": c.json() if c.headers.get('content-type','').startswith('application/json') else c.text[:200]}

asset = upload(VIDEO)
print(json.dumps(asset, indent=2))
SENSOR_ID = asset["sensorId"]; SENSOR_NAME = asset["name"]
print("\nPASS: upload + /complete returned HTTP 200" if asset["complete_status"] == 200 else "FAIL upload")

{
  "sensorId": "1d590bc0-a6c3-4c62-9e7b-a018c80ec4fb",
  "name": "dance_e405af44",
  "upload_status": 200,
  "complete_status": 200,
  "complete_body": {
    "message": "Video dance_e405af44.mp4 successfully uploaded to VST",
    "sensor_id": "1d590bc0-a6c3-4c62-9e7b-a018c80ec4fb",
    "filename": "dance_e405af44.mp4",
    "chunks_processed": 0
  }
}

PASS: upload + /complete returned HTTP 200


## 5. Ingest reached the perception engine
Prove the hand-off worked: VIOS delivered the `camera_streaming` webhook, and `vss-rtvi-cv` accepted `/stream/add`, downloaded the file, and registered the source. Everything up to DeepStream is healthy.

In [6]:
time.sleep(8)  # let the webhook + download happen
logs = rtvi_logs(since="3m")
def has(needle): return any(needle in l for l in logs.splitlines())
print("PASS" if has(SENSOR_ID) or has(SENSOR_NAME) else "NOT SEEN",
      "- rtvi-cv references this sensor")
print("PASS" if has("new stream added") else "NOT SEEN", "- /stream/add accepted")
print("PASS" if has("HTTP_DOWNLOAD") or has("Successfully downloaded") else "(n/a)",
      "- file downloaded into rtvi-cv")
print("\n--- relevant rtvi-cv lines ---")
for l in logs.splitlines():
    if any(k in l for k in ["stream/add", "new stream added", "HTTP_DOWNLOAD", "nvmultiurisrcbin", SENSOR_NAME]):
        print(l)

PASS - rtvi-cv references this sensor
NOT SEEN - /stream/add accepted
PASS - file downloaded into rtvi-cv

--- relevant rtvi-cv lines ---
uri:/api/v1/stream/add
[HTTP_DOWNLOAD] Downloading http://vst-ingress:30888/vst/storage/temp_files/dance_e405af44_20260917_050513_e0860.mp4 to /opt/nvidia/deepstream/deepstream/samples/streams/dance_e405af44_20260917_050513_e0860.mp4
[HTTP_DOWNLOAD] Successfully downloaded to /opt/nvidia/deepstream/deepstream/samples/streams/dance_e405af44_20260917_050513_e0860.mp4
[nvmultiurisrcbin] QUEUED 1d590bc0-a6c3-4c62-9e7b-a018c80ec4fb | active  1/1  [b50e196b-cf65-422e-b7b4-2410a817ddda] | queue  3 [ef3b830c-123f-4b88-b2c4-8179f3da1ca1,ccbfd257-c3c9-4908-add5-9c2fad0d5cfd,1d590bc0-a6c3-4c62-9e7b-a018c80ec4fb]


## 6. THE FAILURE — the source never becomes active, 0 detections emitted
DeepStream has the source and the models, but no frame ever reaches the detector/tracker/Kafka sink. We poll for ~90 s and watch three numbers that should climb but stay pinned at zero:
- **rtvi-cv PERF FPS** (from container logs)
- **`Active sources`** reported by the perception app
- **Kafka `mdx-raw` message count**

If any of these moves off zero, tracking is working. In the failing state, all three stay `0`.

In [7]:
import re
baseline = kafka_offset("mdx-raw")
print(f"{'t(s)':>5} | {'rtvi PERF FPS':>13} | {'Active sources':>14} | {'mdx-raw msgs':>12}")
print("-" * 54)
start = time.time()
while time.time() - start < 90:
    logs = rtvi_logs(since="20s").splitlines()
    perf = next((l for l in reversed(logs) if re.search(r"source_id\s*:\s*0", l)), "")
    m = re.search(r"([0-9]+\.[0-9]+)\s*\(", perf)
    fps = m.group(1) if m else "?"
    act = next((l.split(":")[-1].strip() for l in reversed(logs) if "Active sources" in l), "?")
    msgs = kafka_offset("mdx-raw") - baseline
    print(f"{int(time.time()-start):5d} | {fps:>13} | {act:>14} | {msgs:>12}")
    if msgs > 0:
        print("\n>>> detections are flowing — tracking WORKS <<<"); break
    time.sleep(10)
else:
    print("\n>>> FAIL: 90s elapsed, PERF stayed 0 FPS, Active sources 0, mdx-raw produced 0 messages <<<")

 t(s) | rtvi PERF FPS | Active sources | mdx-raw msgs
------------------------------------------------------
    2 |       0.00000 |              0 |            0
   14 |       0.00000 |              0 |            0


KeyboardInterrupt: 

## 7. The smoking gun in the logs
The decoder receives buffers but they arrive **before the segment event**, so DeepStream drops them and the source never activates. Also shown: the AAC audio-decoder warning (possible trigger — the file's audio pad may stall `uridecodebin` preroll).

In [8]:
logs = rtvi_logs(since="5m").splitlines()
def show(needle, label, n=3):
    hits = [l for l in logs if needle in l]
    print(f"\n[{label}]  ({len(hits)} occurrences)")
    for l in hits[-n:]:
        print("  ", l.strip())
show("Got data flow before segment event", "decoder: data before segment -> dropped")
show("Active sources : 0", "perception: zero active sources")
show("No decoder available", "audio: AAC pad has no decoder (possible preroll stall)")
show("nvmultiurisrcbin", "source manager (note: processes ONE source at a time)")


[decoder: data before segment -> dropped]  (356 occurrences)
   (metropolis_perception_app:1): GStreamer-WARNING **: 05:07:37.811: ../gst/gstpad.c:4463:gst_pad_chain_data_unchecked:<nvv4l2decoder0:sink> Got data flow before segment event
   (metropolis_perception_app:1): GStreamer-WARNING **: 05:07:38.655: ../gst/gstpad.c:4463:gst_pad_chain_data_unchecked:<nvv4l2decoder0:sink> Got data flow before segment event
   (metropolis_perception_app:1): GStreamer-WARNING **: 05:07:39.500: ../gst/gstpad.c:4463:gst_pad_chain_data_unchecked:<nvv4l2decoder0:sink> Got data flow before segment event

[perception: zero active sources]  (60 occurrences)
   Active sources : 0
   Active sources : 0
   Active sources : 0

[audio: AAC pad has no decoder (possible preroll stall)]  (0 occurrences)

[source manager (note: processes ONE source at a time)]  (1 occurrences)
   [nvmultiurisrcbin] QUEUED 1d590bc0-a6c3-4c62-9e7b-a018c80ec4fb | active  1/1  [b50e196b-cf65-422e-b7b4-2410a817ddda] | queue  3 [ef3b830

## 8. Downstream consequence — why the tracking notebook hangs
No CV output -> Kafka `mdx-raw` empty -> Logstash has nothing to index -> ES `mdx-raw-*` is never created. Our original `tracking_walkthrough.ipynb` polls `mdx-raw-*` for the sensor and therefore waits forever.

In [ ]:
print("Kafka mdx-raw total messages :", kafka_offset("mdx-raw"))
rc, out = sh(f"curl -s '{ES_URL}/_cat/indices?h=index,docs.count'")
mdx_idx = [l for l in out.splitlines() if 'mdx-raw' in l]
print("ES mdx-raw-* indices         :", mdx_idx or "(none — never created)")
# What a query from the old tracking notebook would return:
q = {"query": {"term": {"sensorId.keyword": SENSOR_NAME}}}
r = requests.get(f"{ES_URL}/mdx-raw-*/_count", json=q, timeout=10)
print("ES mdx-raw-* count for sensor:", r.status_code, r.text[:200])

## 9. Diagnosis & open questions for NVIDIA

**Confirmed working:** alerts/2d_cv stack healthy; Grounding DINO + ReID tracker loaded; agent upload + `/complete` -> HTTP 200; VIOS `camera_streaming` webhook delivered; `vss-rtvi-cv` accepts `/stream/add`, downloads the file, and registers the source.

**Fails:** the registered source never activates. `vss-rtvi-cv` (DeepStream) logs `PERF 0.00000 FPS` and `Active sources : 0` indefinitely, with continuous `nvv4l2decoder0:sink: Got data flow before segment event`. **Zero** messages are produced to Kafka `mdx-raw`, so ES `mdx-raw-*` is never created.

**Observations for triage:**
1. Buffers reach `nvv4l2decoder` but arrive *before the segment event* and are dropped — the source never reaches PLAYING. Classic caps/segment negotiation stall on the ingested source.
2. The perception app runs **one source at a time** (`active 1/1`); the stuck source blocks the queue, and `POST /api/v1/stream/remove` returns `500 STREAM_REMOVE_FAIL, No record found`, so a wedged source can't be cleared without restarting `vss-rtvi-cv`.
3. The clip has an **AAC audio track** with `No decoder available for audio/mpeg`. Hypothesis: the unlinked audio pad stalls `uridecodebin` preroll. Worth testing whether a **video-only** stream (or the perception pipeline explicitly disabling audio) clears the stall.

**Questions:**
- Is the ingested source expected to be an **RTSP live replay** vs the **HTTP full-file download** that `vss-rtvi-cv` is currently using? Which is the supported 2d_cv path?
- Should the DeepStream pipeline in `vss-rtvi-cv` set `no-audio`/drop the audio pad so AAC files don't stall preroll?
- What is the intended recovery when a source wedges, given `/stream/remove` fails with `No record found`?

> Re-run this notebook top-to-bottom for a clean reproduction. Steps 1-5 PASS, Step 6 FAILS. (`vss-rtvi-cv` may need a restart first if a prior wedged source is still occupying its single slot.)